In [ ]:
# load the complete dataset from csv file
# generate isomap and UMAP values of the sub-groups
# save results as isomap and UMAP csv files for Moving Average generation

# select different sub-groups (eg. ctl and SCA1)
# write out the subject names as csv if needed (eg. ctl_sca1_time12)

In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [9]:
curProject = 'ataxia'
curRoot = 'C'  # 'C' or 'D'

In [217]:
############################  Load information for ATRIL, baseline  #############################
ATRIL_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\SCA_INFO\ATRIL_M0_joy.csv'
try:
    ATRIL_info = pd.read_csv(ATRIL_info_path, index_col=0, header=0,sep=';')
except FileNotFoundError as e:
    print(f"Error: {e}")


ATRIL_info.reset_index(inplace=True)      # Reset index to make old index 'SUBJECT_REF' a column
ATRIL_info['subjID'] = ATRIL_info['SUBJECT_REF'].str.replace('-', '') # Create the new column 'subjName'
ATRIL_info.set_index('subjID', inplace=True)     # Set 'subjID' as the new index
ATRIL_info.rename(columns={'sex': 'Sex', 'age_V1': 'Age', 'SARA_V1': 'SARA','INAS_V1':'INAS','CAGlong':'CAG'}, inplace=True)

# For column 'SCA', remove 'SCA' and keeping the values
ATRIL_info['SCA'] = ATRIL_info['SCA'].str.replace('SCA', '')

# Add column CodeICM: ATRIL
ATRIL_info['CodeICM'] = 'ATRIL'

ATRIL_info.drop(columns=['Inclusion_Date','SUBJECT_REF'],inplace=True)

print("***********  ATRIL INFO  ************")
print(ATRIL_info.head())  
#print(ATRIL_info.columns)
#print(ATRIL_info.index)

***********  ATRIL INFO  ************
          RANDOMIZATION SCA   CAG  Sex        Age  SARA  INAS CodeICM
subjID                                                               
0010001OP             A   2  35.0    1  61.089665   8.0   2.0   ATRIL
0010002MV             B   2  43.0    2  34.858316  11.5   5.0   ATRIL
0010003CJ             B   2  39.0    2  36.522930   7.0   2.0   ATRIL
0010004HV             A   2  43.0    2  32.887064  11.0   4.0   ATRIL
0010005BC             B   2  35.0    2  66.861054  21.0   5.0   ATRIL


In [219]:
############  save info file if needed  ############
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_time1_only.csv'
print(outFileName)
#ATRIL_info.to_csv(outFileName,index=True)

C:\B_projWIP\proj_ataxia\SCA_INFO\processed_INFO\ATRIL_time1_only.csv


In [237]:
####################  Load information for ATRIL, multiple time points  ######################
ATRIL_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\SCA_INFO\ATRIL.csv'

try:
    ATRIL_info = pd.read_csv(ATRIL_info_path, index_col=0, header=0,sep=';')
except FileNotFoundError as e:
    print(f"Error: {e}")

# Reset index to make old index 'SUBJECT_REF' a column, index as integers starting from 0
ATRIL_info.reset_index(inplace=True)

# Create the new column 'subjName'
#ATRIL_info['subjName'] = ATRIL_info['SUBJECT_REF'].str.replace('-', '') + '_M0' # old, for only first time stamp
ATRIL_info['subjName'] = ATRIL_info['SUBJECT_REF'].str.replace('-', '')

# 'CAGlong' and 'Inclusion_Date'  copy to second line
# Group by 'subjName' and use 'transform' to propagate the values of 'CAGlong' and 'Inclusion_Date'
ATRIL_info['CAGlong'] = ATRIL_info.groupby('subjName')['CAGlong'].transform('first')
ATRIL_info['Inclusion_Date'] = ATRIL_info.groupby('subjName')['Inclusion_Date'].transform('first')

# Create the new 'Time_point' column by using 'cumcount' within the groupby operation
ATRIL_info['Time_point'] = ATRIL_info.groupby('subjName').cumcount() + 1
ATRIL_info['Time_point'] = ATRIL_info['Time_point'].astype('int32')

# Update subjName, add _M0 or _M1 according to Time_point value
ATRIL_info['subjName'] = ATRIL_info.apply(lambda row: f"{row['subjName']}_M0" if row['Time_point'] == 1 
                                          else (f"{row['subjName']}_M1" if row['Time_point'] == 2 
                                          else row['subjName']), axis=1)

# Create a boolean mask for even and odd rows, even rows for M0, odd rows for M1, THIS CREATES ERROR when missing time points!!
#is_even_row = np.arange(len(ATRIL_info)) % 2 == 0
# Apply the mask to add '_M0' or '_M1' to even and odd rows to subjName
#ATRIL_info['subjName'] = ATRIL_info['subjName'] + ['_M0' if row % 2 == 0 else '_M1' for row in range(len(ATRIL_info))]

# Set 'subjName' as the new index
ATRIL_info.set_index('subjName', inplace=True)

# For column 'SCA', remove 'SCA' and keeping the values
ATRIL_info['SCA'] = ATRIL_info['SCA'].str.replace('SCA', '')
ATRIL_info['SCA'] = ATRIL_info['SCA'].astype(str)

# Add column CodeICM: ATRIL
ATRIL_info['CodeICM'] = 'ATRIL'

ATRIL_info.drop(columns=['Inclusion_Date','SUBJECT_REF'],inplace=True)

# Rename columns: 
# CAGlong to CAG, sex (1=M, 2=F) to Sex, age_calc to Age, Sara to SARA, inasscore to INAS, RANDOMIZATION (A=placebo, B=riluzole) to Randomization
ATRIL_info = ATRIL_info.rename(columns={'CAGlong':'CAG','sex (1=M, 2=F)':'Sex','age_calc':'Age','Sara':'SARA','inasscore':'INAS',
                                        'RANDOMIZATION (A=placebo, B=riluzole)':'RANDOMIZATION'})


In [239]:
############  save info file if needed  ############
print(ATRIL_info.columns)
print(ATRIL_info.head())
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_time1_2.csv'
print(outFileName)
#ATRIL_info.to_csv(outFileName,index=True)

Index(['RANDOMIZATION', 'SCA', 'CAG', 'Sex', 'Age', 'SARA', 'INAS',
       'Time_point', 'CodeICM'],
      dtype='object')
             RANDOMIZATION SCA   CAG  Sex        Age  SARA  INAS  Time_point  \
subjName                                                                       
0010001OP_M0             A   2  35.0    1  61.089665   8.0   2.0           1   
0010001OP_M1             A   2  35.0    1  62.091718   7.0   2.0           2   
0010002MV_M0             B   2  43.0    2  34.858316  11.5   5.0           1   
0010002MV_M1             B   2  43.0    2  35.871321  10.0   5.0           2   
0010003CJ_M0             B   2  39.0    2  36.522930   7.0   2.0           1   

             CodeICM  
subjName              
0010001OP_M0   ATRIL  
0010001OP_M1   ATRIL  
0010002MV_M0   ATRIL  
0010002MV_M1   ATRIL  
0010003CJ_M0   ATRIL  
C:\B_projWIP\proj_ataxia\SCA_INFO\processed_INFO\ATRIL_time1_2.csv


In [317]:
############################  Load information for BIOSCA, baseline  #############################
BIOSCA_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\SCA_INFO\BIOSCA_joy.csv'
try:
    # Try reading with ISO-8859-1 encoding
    BIOSCA_info = pd.read_csv(BIOSCA_info_path, index_col=0, header=0, encoding='ISO-8859-1')
except UnicodeDecodeError:
    # If ISO-8859-1 fails, try reading with Windows-1252 encoding
    BIOSCA_info = pd.read_csv(BIOSCA_info_path, index_col=0, header=0, encoding='Windows-1252')
except FileNotFoundError as e:
    print(f"Error: {e}")

# List of column names to remove
columns_to_remove = [str(i) for i in range(1, 12)]
BIOSCA_info.drop(columns=columns_to_remove, inplace=True, errors='ignore')

# Reset index to make 'CodeICM' a column
BIOSCA_info.reset_index(inplace=True)
# Create the new column 'subjName'
BIOSCA_info['NinclusionPatient'] = BIOSCA_info['NinclusionPatient'].astype(str).str.zfill(3)
BIOSCA_info['subjID'] = '001' + BIOSCA_info['NinclusionPatient'] + BIOSCA_info['NomPatient'] + BIOSCA_info['PrénomPatient']
BIOSCA_info.set_index('subjID', inplace=True)  

# Rename columns: 
BIOSCA_info = BIOSCA_info.rename(columns={'CAGpatho':'CAG','sex':'Sex','age_start':'Age_onset','Group':'Group_ctl_pre_pat',
                                          'Lateralité':'Handedness','age_V1':'Age','SARA_V1':'SARA','CCFS_V1':'CCFS'})
# Convert to integer
BIOSCA_info['Group_ctl_pre_pat'] = BIOSCA_info['Group_ctl_pre_pat'].astype(int)

# Set SCA type as string
BIOSCA_info['SCA'] = BIOSCA_info['SCA'].astype(str)

# Add the Disease_duration column
BIOSCA_info['Disease_duration'] = BIOSCA_info['Age'] - BIOSCA_info['Age_onset']

BIOSCA_info.drop(columns=['NomPatient','PrénomPatient','NinclusionPatient','age_V2','SARA_V2','CCFS_V2','Group_ctl_pre_pat'],inplace=True)


In [321]:
############  save info file if needed  ############
print(BIOSCA_info.columns)
print(BIOSCA_info.head())
#print(BIOSCA_info)
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\BIOSCA_time1_only.csv'
print(outFileName)
#BIOSCA_info.to_csv(outFileName,index=True)

Index(['CodeICM', 'SCA', 'CAG', 'Sex', 'Age_onset', 'Age', 'SARA', 'CCFS',
       'Handedness', 'Disease_duration'],
      dtype='object')
         CodeICM SCA   CAG  Sex  Age_onset  Age  SARA   CCFS Handedness  \
subjID                                                                    
001001MR  BIOSCA   3  74.0    1       24.0   31   4.0  0.971     Droite   
001005DD  BIOSCA   7  42.0    1       35.0   42   6.0  0.941     Droite   
001006DP  BIOSCA   7  39.0    2       45.0   47   0.0  0.839     Droite   
001007DY  BIOSCA   7  44.0    2       30.0   45   9.5  0.991     Droite   
001008JV  BIOSCA   7  42.0    1       40.0   44  12.0  1.011     Droite   

          Disease_duration  
subjID                      
001001MR               7.0  
001005DD               7.0  
001006DP               2.0  
001007DY              15.0  
001008JV               4.0  
C:\B_projWIP\proj_ataxia\SCA_INFO\processed_INFO\BIOSCA_time1_only.csv


In [339]:
############################  Load information for BIOSCA, multiple time points  #############################
BIOSCA_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\BIOSCA_joy.csv' # added column 'laterality' to BIOSCA_E1_joy, oct 2024
try:
    # Try reading with ISO-8859-1 encoding
    BIOSCA_info = pd.read_csv(BIOSCA_info_path, index_col=0, header=0, encoding='ISO-8859-1')
except UnicodeDecodeError:
    # If ISO-8859-1 fails, try reading with Windows-1252 encoding
    BIOSCA_info = pd.read_csv(BIOSCA_info_path, index_col=0, header=0, encoding='Windows-1252')
except FileNotFoundError as e:
    print(f"Error: {e}")

# List of column names to remove
columns_to_remove = [str(i) for i in range(1, 12)]
BIOSCA_info.drop(columns=columns_to_remove, inplace=True, errors='ignore')

# Reset index to make 'CodeICM' a column
BIOSCA_info.reset_index(inplace=True)
# Create the new column 'subjName'
BIOSCA_info['NinclusionPatient'] = BIOSCA_info['NinclusionPatient'].astype(str).str.zfill(3)
BIOSCA_info['subjName'] = '001' + BIOSCA_info['NinclusionPatient'] + BIOSCA_info['NomPatient'] + BIOSCA_info['PrénomPatient']

# Rename columns: 
BIOSCA_info = BIOSCA_info.rename(columns={'CAGpatho':'CAG','sex':'Sex','age_start':'Age_onset','Group':'Group_ctl_pre_pat','Lateralité':'Handedness'})
# Convert to integer
BIOSCA_info['Group_ctl_pre_pat'] = BIOSCA_info['Group_ctl_pre_pat'].astype(int)

# Set SCA type as string
BIOSCA_info['SCA'] = BIOSCA_info['SCA'].astype(str)


In [341]:
############################  Continue: Load information for BIOSCA, multiple time points  #############################
# Fill up missing values with NaN
BIOSCA_info[['CAG', 'Age_onset']] = BIOSCA_info[['CAG', 'Age_onset']].fillna(np.nan)

# 1. Split the columns that contain stable information and those that change over time (like _V1, _V2)
stable_columns = ['subjName','SCA','CAG', 'Sex', 'Age_onset','Group_ctl_pre_pat','CodeICM','Handedness']
changing_columns = ['age_V1','age_V2','SARA_V1','SARA_V2','CCFS_V1','CCFS_V2']

# 2. Reshape (melt) time-varying columns. Use pd.melt() to "unpivot" the _V1 and _V2 columns into rows
melted_df = BIOSCA_info.melt(id_vars=stable_columns, value_vars=changing_columns, 
                    var_name='MeasureType', value_name='Value')

# 3. Extract time point information (_V1 or _V2) from 'MeasureType' and create 'Time_point'
melted_df['Time_point'] = melted_df['MeasureType'].apply(lambda x: 'V1' if '_V1' in x else 'V2')

# 4. Clean up 'MeasureType' to remove _V1/_V2 suffix
melted_df['MeasureType'] = melted_df['MeasureType'].str.replace('_V1', '').str.replace('_V2', '')

# 5. Pivot back to get each measure on its own column for each time point
# This doesn't work since pivot_table removes NaN values, use pivot instead
#final_df = melted_df.pivot_table(index=stable_columns + ['Time_point'], 
#                                 columns='MeasureType', values='Value').reset_index()
final_df = melted_df.pivot(index=stable_columns + ['Time_point'], columns='MeasureType', values='Value').reset_index()

# For column 'Time_point', remove 'V' and keeping the values
final_df['Time_point'] = final_df['Time_point'].str.replace('V', '')
final_df = final_df.rename(columns={'age':'Age'})

# Convert Time_point to integer
final_df['Time_point'] = final_df['Time_point'].astype(int)
# Update subjName, add _M0 or _M1 according to Time_point value
final_df['subjName'] = final_df.apply(lambda row: f"{row['subjName']}_E1" if row['Time_point'] == 1 
                                          else (f"{row['subjName']}_E2" if row['Time_point'] == 2 
                                          else row['subjName']), axis=1)
BIOSCA_info_melt = final_df.copy()

# Set subjName as index
BIOSCA_info_melt.set_index('subjName', inplace=True)

# Add the Disease_duration column
BIOSCA_info_melt['Disease_duration'] = BIOSCA_info_melt['Age'] - BIOSCA_info_melt['Age_onset']

BIOSCA_info = BIOSCA_info_melt

# remove unnecessary columns
BIOSCA_info.drop(columns=['Group_ctl_pre_pat'],inplace=True)

In [345]:
############  save info file if needed  ############
print(BIOSCA_info.columns)
print(BIOSCA_info.head())
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\BIOSCA_time1_2.csv'
print(outFileName)
#BIOSCA_info.to_csv(outFileName,index=True)

Index(['SCA', 'CAG', 'Sex', 'Age_onset', 'CodeICM', 'Handedness', 'Time_point',
       'CCFS', 'SARA', 'Age', 'Disease_duration'],
      dtype='object', name='MeasureType')
MeasureType SCA   CAG  Sex  Age_onset CodeICM Handedness  Time_point   CCFS  \
subjName                                                                      
001001MR_E1   3  74.0    1       24.0  BIOSCA     Droite           1  0.971   
001001MR_E2   3  74.0    1       24.0  BIOSCA     Droite           2  0.940   
001005DD_E1   7  42.0    1       35.0  BIOSCA     Droite           1  0.941   
001005DD_E2   7  42.0    1       35.0  BIOSCA     Droite           2  0.891   
001006DP_E1   7  39.0    2       45.0  BIOSCA     Droite           1  0.839   

MeasureType  SARA   Age  Disease_duration  
subjName                                   
001001MR_E1   4.0  31.0               7.0  
001001MR_E2   6.0  33.0               9.0  
001005DD_E1   6.0  42.0               7.0  
001005DD_E2   5.0  44.0               9.0  
001006DP_

In [275]:
############################  Load information for CERMOI, baseline  #############################
CERMOI_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\SCA_INFO\CERMOI_V1_joy.csv'

try:
    # Try reading with ISO-8859-1 encoding
    CERMOI_info = pd.read_csv(CERMOI_info_path, index_col=0, header=0, encoding='ISO-8859-1')
except UnicodeDecodeError:
    # If ISO-8859-1 fails, try reading with Windows-1252 encoding
    CERMOI_info = pd.read_csv(CERMOI_info_path, index_col=0, header=0, encoding='Windows-1252')
except FileNotFoundError as e:
    print(f"Error: {e}")

# Reset index to make 'ID' a column
CERMOI_info.reset_index(inplace=True) 
CERMOI_info.rename(columns={'ï»¿ID': 'ID'}, inplace=True)  # to remove unknown characters from the column name

# Transform the 'participant_id' column
CERMOI_info['subjName'] = CERMOI_info['participant_id'].str.replace('ICM', '')  # Remove 'ICM'
CERMOI_info['subjName'] = CERMOI_info['subjName'].str.replace('-', '')    # Remove all '-'
CERMOI_info['subjName'] = '00' + CERMOI_info['subjName']                   # Add '00' prefix

# Define the mapping from Group_CERMOI to SCA
group_to_sca = {1: '2', 2: '7', 3: '0'}
# Create the new column 'SCA' based on the mapping
CERMOI_info['SCA'] = CERMOI_info['Group_CERMOI'].map(group_to_sca)

# Add column CodeICM: CERMOI
CERMOI_info['CodeICM'] = 'CERMOI'

# Convert 'allele1' and 'allele2' to numeric, coercing errors to NaN
CERMOI_info['allele1'] = pd.to_numeric(CERMOI_info['allele1'], errors='coerce')
CERMOI_info['allele2'] = pd.to_numeric(CERMOI_info['allele2'], errors='coerce')

# Add columns minAllele, maxAllele and sumAllele
CERMOI_info['minAllele'] = CERMOI_info[['allele1', 'allele2']].min(axis=1)
CERMOI_info['maxAllele'] = CERMOI_info[['allele1', 'allele2']].max(axis=1)
CERMOI_info['sumAllele'] = CERMOI_info[['allele1', 'allele2']].sum(axis=1, skipna=False) # keep NaN if allele 1/2 is NaN

# Add a column CAG, copy maxAllele
CERMOI_info['CAG'] = CERMOI_info['maxAllele']

CERMOI_info = CERMOI_info.rename(columns={'sex':'Sex','age_V1':'Age','age_start':'Age_onset',
                                          'INAS_V1':'INAS','SARA_V1':'SARA','subjName':'subjID','Disease_Duration':'Disease_duration'})
CERMOI_info.set_index('subjID', inplace=True)  

CERMOI_info.drop(columns=['ID','Visit ','participant_id','visitdate','birthyear','Group_CERMOI'],inplace=True)
CERMOI_info[['Age_onset', 'Disease_duration']] = CERMOI_info[['Age_onset', 'Disease_duration']].replace(' ', np.nan)
print(CERMOI_info.columns)

Index(['allele1', 'allele2', 'Sex', 'Age', 'Age_onset', 'Disease_duration',
       'INAS', 'SARA', 'SCA', 'CodeICM', 'minAllele', 'maxAllele', 'sumAllele',
       'CAG'],
      dtype='object')


In [279]:
############  save info file if needed  ############
print(CERMOI_info.head())
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\CERMOI_time1_only.csv'
print(outFileName)
#CERMOI_info.to_csv(outFileName,index=True)

         allele1  allele2  Sex  Age Age_onset Disease_duration  INAS  SARA  \
subjID                                                                       
00001PJ     22.0     22.0    2   31       NaN              NaN     1   0.0   
00002PV     22.0     40.0    1   32       NaN              NaN     2   1.5   
00003OA     12.0     46.0    2   38        36                2     5  12.0   
00004PA     22.0     22.0    1   28       NaN              NaN     2   0.0   
00005PS     10.0     62.0    2   18         6               12     7  15.0   

        SCA CodeICM  minAllele  maxAllele  sumAllele   CAG  
subjID                                                      
00001PJ   0  CERMOI       22.0       22.0       44.0  22.0  
00002PV   2  CERMOI       22.0       40.0       62.0  40.0  
00003OA   7  CERMOI       12.0       46.0       58.0  46.0  
00004PA   0  CERMOI       22.0       22.0       44.0  22.0  
00005PS   7  CERMOI       10.0       62.0       72.0  62.0  
C:\B_projWIP\proj_ataxia\S

In [301]:
############################  Load information for CERMOI, multiple time points  #############################
# Load information for CERMOI
CERMOI_info_path = rf'{curRoot}:\B_projWIP\proj_ataxia\SCA_INFO\CERMOI_joy.csv'

try:
    # Try reading with ISO-8859-1 encoding
    CERMOI_info = pd.read_csv(CERMOI_info_path, index_col=0, header=0, encoding='ISO-8859-1')
except UnicodeDecodeError:
    # If ISO-8859-1 fails, try reading with Windows-1252 encoding
    CERMOI_info = pd.read_csv(CERMOI_info_path, index_col=0, header=0, encoding='Windows-1252')
except FileNotFoundError as e:
    print(f"Error: {e}")

# Reset index to make 'ID' a column
CERMOI_info.reset_index(inplace=True) 
CERMOI_info.rename(columns={'ï»¿ID': 'ID'}, inplace=True)  # to remove unknown characters from the column name

# Transform the 'participant_id' column
CERMOI_info['subjName'] = CERMOI_info['participant_id'].str.replace('ICM', '')  # Remove 'ICM'
CERMOI_info['subjName'] = CERMOI_info['subjName'].str.replace('-', '')    # Remove all '-'
CERMOI_info['subjName'] = '00' + CERMOI_info['subjName']                   # Add '00' prefix
#CERMOI_info['subjName'] = CERMOI_info['subjName'] + '_V1'                  # Add '_V1' postfix

# Define the mapping from Group_CERMOI to SCA
group_to_sca = {1: '2', 2: '7', 3: '0'}
# Create the new column 'SCA' based on the mapping
#CERMOI_info['SCA'] = CERMOI_info['Group_CERMOI'].map(group_to_sca)
CERMOI_info['SCA'] = CERMOI_info['group (1=SCA2, 2=SCA7, 3=control)'].map(group_to_sca)

# Add column CodeICM: CERMOI
CERMOI_info['CodeICM'] = 'CERMOI'

# Convert 'allele1' and 'allele2' to numeric, coercing errors to NaN
CERMOI_info['allele1'] = pd.to_numeric(CERMOI_info['allele1'], errors='coerce')
CERMOI_info['allele2'] = pd.to_numeric(CERMOI_info['allele2'], errors='coerce')

# Add columns minAllele, maxAllele and sumAllele
CERMOI_info['minAllele'] = CERMOI_info[['allele1', 'allele2']].min(axis=1)
CERMOI_info['maxAllele'] = CERMOI_info[['allele1', 'allele2']].max(axis=1)
CERMOI_info['sumAllele'] = CERMOI_info[['allele1', 'allele2']].sum(axis=1, skipna=False) # keep NaN if allele 1/2 is NaN


# Add a column CAG, copy maxAllele
CERMOI_info['CAG'] = CERMOI_info['maxAllele']

# Rename columns: 
CERMOI_info = CERMOI_info.rename(columns={'gender':'Sex','Age at visit ':'Age','Age of first symptom':'Age_onset',
                                          'Disease duration ':'Disease_duration','inas_count':'INAS'})

In [303]:
# copy to second and third line, group by 'ID' and use 'transform' to propagate the values
CERMOI_info['Sex'] = CERMOI_info.groupby('ID')['Sex'].transform('first')
CERMOI_info['Age_onset'] = CERMOI_info.groupby('ID')['Age_onset'].transform('first')
CERMOI_info['Disease_duration'] = CERMOI_info.groupby('ID')['Disease_duration'].transform('first')
CERMOI_info['subjName'] = CERMOI_info.groupby('ID')['subjName'].transform('first')
CERMOI_info['SCA'] = CERMOI_info.groupby('ID')['SCA'].transform('first')
CERMOI_info['CodeICM'] = CERMOI_info.groupby('ID')['CodeICM'].transform('first')
CERMOI_info['CAG'] = CERMOI_info.groupby('ID')['CAG'].transform('first')
CERMOI_info['allele1'] = CERMOI_info.groupby('ID')['allele1'].transform('first')
CERMOI_info['allele2'] = CERMOI_info.groupby('ID')['allele2'].transform('first')
CERMOI_info['minAllele'] = CERMOI_info.groupby('ID')['minAllele'].transform('first')
CERMOI_info['maxAllele'] = CERMOI_info.groupby('ID')['maxAllele'].transform('first')
CERMOI_info['sumAllele'] = CERMOI_info.groupby('ID')['sumAllele'].transform('first')
CERMOI_info['participant_id'] = CERMOI_info.groupby('ID')['participant_id'].transform('first')

########### add 0.5 and 1 to the second and third rows of Age for each ID ###########
# Function to propagate Age with offsets
def propagate_age(group):
    # Get the base age
    base_age = group['Age'].values
    # Create new values with the specified offsets
    new_ages = [base_age[0], base_age[0] + 0.5, base_age[0] + 1]
    # Return new ages with NaNs for other IDs
    return pd.Series(new_ages + [None] * (len(group) - 3), index=group.index)
# Apply the function to each group and fill in the Age column
CERMOI_info['Age'] = CERMOI_info.groupby('ID').apply(propagate_age).reset_index(drop=True)

# remove columns with redundant information
columns_to_remove = ['group (1=SCA2, 2=SCA7, 3=control)', 'birthyear','Visit ','visitdate']
CERMOI_info = CERMOI_info.drop(columns=columns_to_remove)

##############################  checking empty rows  #############################
#print(CERMOI_info['Age_onset'].unique())
#print(CERMOI_info['Disease_duration'].unique())

# Replace empty strings and other placeholders with np.nan if needed
CERMOI_info[['Age_onset', 'Disease_duration']] = CERMOI_info[['Age_onset', 'Disease_duration']].replace(' ', np.nan)

# Then, use pd.to_numeric with apply to convert non-numeric values to NaN
CERMOI_info['Age_onset'] = CERMOI_info['Age_onset'].apply(pd.to_numeric, errors='coerce')
CERMOI_info['Disease_duration'] = CERMOI_info['Disease_duration'].apply(pd.to_numeric, errors='coerce')

C:\Users\joyca\AppData\Local\Temp\ipykernel_33116\1095864917.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CERMOI_info['Age'] = CERMOI_info.groupby('ID').apply(propagate_age).reset_index(drop=True)


In [305]:
# Create the new 'Time_point' column by using 'cumcount' within the groupby operation
CERMOI_info['Time_point'] = CERMOI_info.groupby('ID').cumcount() + 1
CERMOI_info['Time_point'] = CERMOI_info['Time_point'].astype('int32')

# Update subjName, add _M0 or _M1 according to Time_point value
CERMOI_info['subjName'] = CERMOI_info.apply(
    lambda row: f"{row['subjName']}_V1" if row['Time_point'] == 1 
    else (f"{row['subjName']}_V2" if row['Time_point'] == 2 
    else (f"{row['subjName']}_V3" if row['Time_point'] == 3 
    else row['subjName'])),
    axis=1
)
# Set 'subjName' as the new index
CERMOI_info.set_index('subjName', inplace=True)

# remove unnecessary columns
CERMOI_info.drop(columns=['participant_id','ID'],inplace=True)

CERMOI_info['SCA'] = CERMOI_info['SCA'].astype(str)
CERMOI_info['Sex'] = CERMOI_info['Sex'].astype('int64')
CERMOI_info['INAS'] = CERMOI_info['INAS'].astype('float64')
CERMOI_info['SARA'] = CERMOI_info['SARA'].astype('float64')
CERMOI_info['CodeICM'] = 'CERMOI'

In [311]:
############  save info file if needed  ############
print(CERMOI_info.columns)
print(CERMOI_info)
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\CERMOI_time1_2.csv'
print(outFileName)
#CERMOI_info.to_csv(outFileName,index=True)

Index(['allele1', 'allele2', 'Sex', 'Age', 'Age_onset', 'Disease_duration',
       'INAS', 'SARA', 'SCA', 'CodeICM', 'minAllele', 'maxAllele', 'sumAllele',
       'CAG', 'Time_point'],
      dtype='object')
            allele1  allele2  Sex   Age  Age_onset  Disease_duration  INAS  \
subjName                                                                     
00001PJ_V1     22.0     22.0    2  31.0        NaN               NaN   1.0   
00001PJ_V2     22.0     22.0    2  31.5        NaN               NaN   1.0   
00001PJ_V3     22.0     22.0    2  32.0        NaN               NaN   1.0   
00002PV_V1     22.0     40.0    1  32.0        NaN               NaN   2.0   
00002PV_V2     22.0     40.0    1  32.5        NaN               NaN   2.0   
...             ...      ...  ...   ...        ...               ...   ...   
00039OV_V2     22.0     36.0    2  40.5        NaN               NaN   2.0   
00039OV_V3     22.0     36.0    2  41.0        NaN               NaN   2.0   
00040DH_V1   

In [347]:
#####################################  joining ATRIL, BIOSCA, CERMOI info  ##############################
ATRIL_baseline_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_time1_only.csv'
ATRIL_complete_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_time1_2.csv'
BIOSCA_baseline_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\BIOSCA_time1_only.csv'
BIOSCA_complete_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\BIOSCA_time1_2.csv'
CERMOI_baseline_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\CERMOI_time1_only.csv'
CERMOI_complete_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\CERMOI_time1_2.csv'


try:
    ATRIL_baseline = pd.read_csv(ATRIL_baseline_info_path, index_col=0, header=0)
    ATRIL_complete = pd.read_csv(ATRIL_complete_info_path, index_col=0, header=0)
    BIOSCA_baseline = pd.read_csv(BIOSCA_baseline_info_path, index_col=0, header=0)
    BIOSCA_complete = pd.read_csv(BIOSCA_complete_info_path, index_col=0, header=0)
    CERMOI_baseline = pd.read_csv(CERMOI_baseline_info_path, index_col=0, header=0)
    CERMOI_complete = pd.read_csv(CERMOI_complete_info_path, index_col=0, header=0)
except FileNotFoundError as e:
    print(f"Error: {e}")

merged_baseline = pd.concat([ATRIL_baseline, BIOSCA_baseline, CERMOI_baseline], axis=0, sort=False)
merged_complete = pd.concat([ATRIL_complete, BIOSCA_complete, CERMOI_complete], axis=0, sort=False)

In [353]:
############  save info file if needed  ############
print(merged_baseline.columns)
print(merged_complete.columns)
outBaselineName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_only.csv'
outCompleteName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_2.csv'
print(outBaselineName)
print(outCompleteName)
#merged_baseline.to_csv(outBaselineName,index=True)
#merged_complete.to_csv(outCompleteName,index=True)

Index(['RANDOMIZATION', 'SCA', 'CAG', 'Sex', 'Age', 'SARA', 'INAS', 'CodeICM',
       'Age_onset', 'CCFS', 'Handedness', 'Disease_duration', 'allele1',
       'allele2', 'minAllele', 'maxAllele', 'sumAllele'],
      dtype='object')
Index(['RANDOMIZATION', 'SCA', 'CAG', 'Sex', 'Age', 'SARA', 'INAS',
       'Time_point', 'CodeICM', 'Age_onset', 'Handedness', 'CCFS',
       'Disease_duration', 'allele1', 'allele2', 'minAllele', 'maxAllele',
       'sumAllele'],
      dtype='object')
C:\B_projWIP\proj_ataxia\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_only.csv
C:\B_projWIP\proj_ataxia\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_2.csv
